# Formula One Racing: Analyzing Driver Performance and Success Factors

**Project Members:** [Your Names Here]

**Date:** December 2024

## 1. Introduction

Formula One (F1) is the highest class of international racing for open-wheel single-seater formula racing cars. This project analyzes Formula One racing data to understand the factors that contribute to driver success. Our main research questions are:

1. Which countries produce the most successful F1 drivers?
2. How has driver performance evolved over the years?
3. What is the relationship between starting grid position and final race position?

Using data from Formula One races spanning multiple decades, we will merge driver information with race results to uncover patterns and insights. Our analysis reveals that nationality, grid position, and era all play significant roles in determining racing success. The following sections detail our data sources, methodology, and key findings.

## 2. Data Description

### 2.1 Dataset Overview

We are using the Formula One dataset from Kaggle, which contains comprehensive data on F1 races from 1950 to the present. For this analysis, we will focus on three main tables:

1. **drivers.csv**: Contains information about each F1 driver including their name, nationality, and date of birth. Each row represents a unique driver.

2. **results.csv**: Contains race results with information about finishing position, points earned, grid position, and lap times. Each row represents a driver's performance in a specific race.

3. **races.csv**: Contains information about each race including the year, circuit, and race name. Each row represents a unique race event.

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Load the datasets
drivers = pd.read_csv("1-Formula_One/drivers.csv")
results = pd.read_csv("1-Formula_One/results.csv")
races = pd.read_csv("1-Formula_One/races.csv")

In [ ]:
# Display basic information about each dataset
print("Drivers Dataset:")
print(f"Number of rows: {len(drivers)}")
print(f"Number of columns: {len(drivers.columns)}")
print(f"Columns: {list(drivers.columns)}")
print()

print("Results Dataset:")
print(f"Number of rows: {len(results)}")
print(f"Number of columns: {len(results.columns)}")
print(f"Columns: {list(results.columns)}")
print()

print("Races Dataset:")
print(f"Number of rows: {len(races)}")
print(f"Number of columns: {len(races.columns)}")
print(f"Columns: {list(races.columns)}")

### 2.2 Data Merging

To analyze driver performance comprehensively, we need to merge the three datasets. The merge process works as follows:
- First, we merge `results` with `drivers` on `driverId` to get driver information for each result
- Then, we merge with `races` on `raceId` to get race year and name information

In [ ]:
# Merge results with drivers
results_drivers = pd.merge(results, drivers, on="driverId", how="left")

# Merge with races to get race year
df = pd.merge(results_drivers, races[["raceId", "year", "name"]], on="raceId", how="left")

print(f"Merged dataset has {len(df)} rows and {len(df.columns)} columns")
df.head()

### 2.3 Data Cleaning

We need to clean the data by:
1. Converting position columns to numeric values
2. Handling missing values represented as '\N'
3. Creating a full name column for easier identification

In [ ]:
# Create a function to clean position data
def clean_position(pos):
    """Convert position to numeric, handling special cases."""
    if pd.isna(pos) or pos == "\\N":
        return np.nan
    try:
        return int(pos)
    except (ValueError, TypeError):
        return np.nan

# Apply cleaning to position column
df["position_clean"] = df["position"].apply(clean_position)

# Create full name column
df["full_name"] = df["forename"] + " " + df["surname"]

# Convert grid position to numeric
df["grid_clean"] = pd.to_numeric(df["grid"], errors="coerce")

print(f"Data cleaning complete!")
print(f"Rows with valid finish position: {df['position_clean'].notna().sum()}")
print(f"Rows with missing finish position: {df['position_clean'].isna().sum()}")

### 2.4 Column Descriptions

Key columns in our merged dataset:

In [ ]:
# Select key columns for analysis
key_columns = ["full_name", "nationality", "year", "points", "position_clean", "grid_clean"]

# Compute descriptive statistics
descriptive_stats = df[key_columns].describe()
print("Descriptive Statistics for Key Columns:")
descriptive_stats

## 3. Results

### 3.1 Which Countries Produce the Best Drivers?

We analyze total wins and points by nationality to determine which countries have produced the most successful F1 drivers.

In [ ]:
# Filter for wins (position = 1)
wins = df[df["position_clean"] == 1]

# Count wins by nationality
wins_by_nationality = wins.groupby("nationality").size().reset_index(name="total_wins")
wins_by_nationality = wins_by_nationality.sort_values("total_wins", ascending=False)

# Display top 10 countries
print("Top 10 Countries by Total Wins:")
wins_by_nationality.head(10)

In [ ]:
# Create a bar chart for wins by nationality
plt.figure(figsize=(12, 6))
top_10_countries = wins_by_nationality.head(10)
plt.bar(top_10_countries["nationality"], top_10_countries["total_wins"], color="steelblue")
plt.xlabel("Nationality", fontsize=12)
plt.ylabel("Total Wins", fontsize=12)
plt.title("Formula One Wins by Driver Nationality (Top 10 Countries)", fontsize=14)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

### 3.2 Total Points by Nationality

In [ ]:
# Calculate total points by nationality
points_by_nationality = df.groupby("nationality")["points"].sum().reset_index()
points_by_nationality = points_by_nationality.sort_values("points", ascending=False)

# Display top 10 countries by points
print("Top 10 Countries by Total Points:")
points_by_nationality.head(10)

In [ ]:
# Create a bar chart for points by nationality
plt.figure(figsize=(12, 6))
top_10_points = points_by_nationality.head(10)
plt.bar(top_10_points["nationality"], top_10_points["points"], color="darkorange")
plt.xlabel("Nationality", fontsize=12)
plt.ylabel("Total Points", fontsize=12)
plt.title("Formula One Points by Driver Nationality (Top 10 Countries)", fontsize=14)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

### 3.3 Driver Performance Over Time

We analyze how average points per race have changed over the years.

In [ ]:
# Calculate average points per year
avg_points_by_year = df.groupby("year")["points"].mean().reset_index()

# Create a line plot
plt.figure(figsize=(14, 6))
plt.plot(avg_points_by_year["year"], avg_points_by_year["points"], marker="o", markersize=3, linewidth=1.5, color="green")
plt.xlabel("Year", fontsize=12)
plt.ylabel("Average Points per Race", fontsize=12)
plt.title("Average Points per Race Over Time in Formula One", fontsize=14)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 3.4 Grid Position vs. Finishing Position

We analyze the relationship between starting grid position and final race position to understand the importance of qualifying.

In [ ]:
# Filter for valid grid and position data
grid_position_data = df[(df["grid_clean"].notna()) & (df["position_clean"].notna())].copy()

# Calculate average finish position for each grid position
grid_vs_finish = grid_position_data.groupby("grid_clean")["position_clean"].mean().reset_index()
grid_vs_finish.columns = ["grid_position", "avg_finish_position"]

# Filter for reasonable grid positions (1-25)
grid_vs_finish = grid_vs_finish[grid_vs_finish["grid_position"] <= 25]

# Create scatter plot with trend line
plt.figure(figsize=(10, 8))
plt.scatter(grid_vs_finish["grid_position"], grid_vs_finish["avg_finish_position"], s=100, color="purple", alpha=0.7)
plt.plot([0, 25], [0, 25], "--", color="gray", label="If finish = grid")
plt.xlabel("Grid (Starting) Position", fontsize=12)
plt.ylabel("Average Finishing Position", fontsize=12)
plt.title("Grid Position vs. Average Finishing Position", fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 3.5 Top Drivers Analysis

We use a loop and custom function to analyze the top 10 drivers by total wins.

In [ ]:
# Function to calculate driver statistics
def calculate_driver_stats(driver_name, data):
    """
    Calculate statistics for a specific driver.
    
    Parameters:
    - driver_name: The full name of the driver
    - data: The merged DataFrame
    
    Returns:
    - Dictionary with driver statistics
    """
    driver_data = data[data["full_name"] == driver_name]
    
    stats = {
        "name": driver_name,
        "total_races": len(driver_data),
        "total_wins": len(driver_data[driver_data["position_clean"] == 1]),
        "total_points": driver_data["points"].sum(),
        "avg_finish": driver_data["position_clean"].mean(),
        "nationality": driver_data["nationality"].iloc[0] if len(driver_data) > 0 else "Unknown"
    }
    
    return stats

In [ ]:
# Get top 10 drivers by wins using a loop
wins_by_driver = df[df["position_clean"] == 1].groupby("full_name").size().reset_index(name="wins")
wins_by_driver = wins_by_driver.sort_values("wins", ascending=False)
top_10_drivers = wins_by_driver.head(10)["full_name"].tolist()

# Calculate statistics for each top driver using a loop
top_driver_stats = []
for driver in top_10_drivers:
    stats = calculate_driver_stats(driver, df)
    top_driver_stats.append(stats)

# Convert to DataFrame for display
top_drivers_df = pd.DataFrame(top_driver_stats)
print("Top 10 Drivers by Wins:")
top_drivers_df

In [ ]:
# Create a horizontal bar chart for top drivers
plt.figure(figsize=(12, 8))
colors = plt.cm.viridis(np.linspace(0, 0.8, len(top_drivers_df)))
plt.barh(top_drivers_df["name"], top_drivers_df["total_wins"], color=colors)
plt.xlabel("Total Wins", fontsize=12)
plt.ylabel("Driver", fontsize=12)
plt.title("Top 10 Formula One Drivers by Total Wins", fontsize=14)
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

### 3.6 Performance by Decade

We analyze how the sport has evolved by grouping races into decades.

In [ ]:
# Create decade column using apply
def get_decade(year):
    """Return the decade for a given year."""
    return (year // 10) * 10

df["decade"] = df["year"].apply(get_decade)

# Calculate statistics by decade
decade_stats = df.groupby("decade").agg({
    "raceId": "nunique",
    "driverId": "nunique",
    "points": "mean"
}).reset_index()

decade_stats.columns = ["Decade", "Unique Races", "Unique Drivers", "Avg Points per Race"]
print("Statistics by Decade:")
decade_stats

In [ ]:
# Create a grouped bar chart showing decade statistics
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Number of races per decade
axes[0].bar(decade_stats["Decade"].astype(str), decade_stats["Unique Races"], color="teal")
axes[0].set_xlabel("Decade", fontsize=12)
axes[0].set_ylabel("Number of Races", fontsize=12)
axes[0].set_title("F1 Races per Decade", fontsize=14)
axes[0].tick_params(axis="x", rotation=45)

# Number of unique drivers per decade
axes[1].bar(decade_stats["Decade"].astype(str), decade_stats["Unique Drivers"], color="coral")
axes[1].set_xlabel("Decade", fontsize=12)
axes[1].set_ylabel("Number of Unique Drivers", fontsize=12)
axes[1].set_title("Unique F1 Drivers per Decade", fontsize=14)
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

## 4. Discussion

Our analysis of Formula One racing data reveals several interesting findings:

**Key Findings:**
1. **Nationality and Success**: British drivers have historically dominated Formula One in terms of total wins, followed by German and Brazilian drivers. This reflects both the historical importance of British motorsport culture and the success of specific drivers like Lewis Hamilton, Jim Clark, and Nigel Mansell.

2. **Grid Position Importance**: There is a strong correlation between starting grid position and final race position. Drivers who qualify in the top positions are significantly more likely to finish near the front, highlighting the importance of qualifying performance in F1.

3. **Evolution of the Sport**: The number of races and participating drivers has increased significantly over the decades, reflecting the global expansion of Formula One. The points system has also changed over time, affecting the average points per race.

4. **Top Drivers**: The most successful drivers in F1 history (by wins) include legends like Lewis Hamilton, Michael Schumacher, and Ayrton Senna. These drivers combine exceptional talent, consistency, and often the advantage of competitive cars.

These findings provide insights into the factors that contribute to success in Formula One racing and how the sport has evolved since its inception in 1950.